# Stand-alone notebook for registering the FlyWire circuit

Adapted from `examples/L_utils/register_circuit.ipynb` for the FlyWire
*Drosophila* whole-brain **point-neuron** model.

- Builds the SONATA circuit (if not already present)
- Registers the circuit entity, its links, and all assets via the high-level
  `register_circuit_from_metadata` helper from `obi_one`

Counts, scale, and circuit properties (morphologies, point neurons, electrical
models, spines) are computed automatically from the SONATA files, and all
additional assets are generated and uploaded by the helper.

## Circuit information

General metadata for the FlyWire model, provided as a dict.

**IMPORTANT:**
- **The circuit name must not yet exist in entitycore (to avoid duplicates).**
- **Linked entities must already exist in entitycore (otherwise they must first
  be registered):** species, subject, brain region hierarchy, brain region, and
  license.

Counts (neurons/synapses/connections), scale, and circuit properties are now
computed automatically from the SONATA files — they no longer need to be
provided here.

In [1]:
from entitysdk.types import CircuitBuildCategory, TargetSimulator

circuit_metadata = {
    # Name of the circuit [Must not yet exist in entitycore, to avoid duplicates]
    "name": "FlyWire Test 3",

    # Description of the circuit
    "description": (
        "FlyWire whole-brain Drosophila point-neuron model (SONATA), "
        "converted with examples/J_drosophila/drosophila_to_brian2_sonata.py"
    ),

    # Circuit build category (computational_model, em_reconstruction).
    # LIF point-neuron simulation model -> computational_model.
    "build_category": CircuitBuildCategory.computational_model,

    # Target simulator (NEURON, Brian2, ...) [Must be consistent with circuit config]
    "target_simulator": TargetSimulator.Brian2,

    # --- Linked entities (must already exist in entitycore) ---
    # NOTE: The values below are *rat placeholders* (matching
    # examples/L_utils/register_circuit.ipynb) so the registration flow can be
    # tested while the Drosophila entities are not yet registered. Swap in the
    # commented Drosophila values once the corresponding species / subject /
    # brain region hierarchy / brain region exist in entitycore.

    # # Species name
    # "species": "Drosophila melanogaster",
    # # Subject name [its species must equal `species` above]
    # "subject": "<Drosophila subject name>",
    # # Brain region hierarchy name for the given species
    # "brain_region_hierarchy": "<Drosophila brain region hierarchy>",
    # # Brain region name within the hierarchy
    # "brain_region": "<Drosophila whole-brain region>",

    # Species name [Must exist in entitycore; otherwise, must first be registered]
    "species": "Rattus norvegicus",

    # Subject name [Must exist in entitycore; its species must equal `species`]
    "subject": "Average rat P14",

    # Brain region hierarchy name for the given species [Must exist in entitycore]
    "brain_region_hierarchy": (
        "Waxholm Space atlas of the rat brain with layers and hippocampal subregions"
    ),

    # Brain region name within the hierarchy [Must exist in entitycore]
    "brain_region": "Primary somatosensory area, hindlimb representation",

    # --- Optional metadata (set to None if not applicable) ---
    # Name of the root circuit (top parent in the derivation hierarchy), if any
    "root": None,

    # Name of the parent circuit (direct parent it was extracted/derived from)
    "parent": None,

    # Type of circuit derivation (circuit_extraction, circuit_rewiring), if parent given
    "derivation_type": None,

    # License name [Must exist in entitycore if not None]
    "license": None,

    # Short, human readable publication string
    "published_in": None,

    # Contact e-mail address
    "contact": None,

    # Experiment date ("November, 2024" -> 01.11.2024, or "27.03.2024")
    "experiment_date": None,
}

### Optional: Publications

DOI -> {"type": entity_source | component_source | application}. Each DOI must
already be registered in entitycore. Empty for *FlyWire Test*.

In [2]:
circuit_publications = {}

### Optional: Contributors

Agent name -> {"type": person | organization | consortium, "role": ...}. Each
agent must already exist in entitycore. Empty for *FlyWire Test*.

In [3]:
circuit_contributions = {}

# Install Brian2

### Circuit file paths

The FlyWire SONATA circuit is built locally (if not already present) with
`drosophila_to_brian2_sonata.convert()`, and `circuit_path` is set to that
folder (it must contain `circuit_config.json` and `node_sets.json`).

`circuit_path` may be one of:
- A folder containing `circuit_config.json`
- A direct path to the `circuit_config.json` file
- A compressed `.gz` archive of the circuit folder

Overview / sim-designer images are optional — if not provided they are
generated automatically. All other additional assets (compressed circuit,
connectivity matrices, connectivity plots) are always generated and uploaded
by the helper.

In [4]:
!uv pip install brian2

Using Python 3.12.9 environment at: /Users/james/Documents/obi/code/obi-main/obi-one/.venv
Checked 1 package in 6ms


In [5]:
import subprocess
import sys
import urllib.request
from pathlib import Path

HERE = Path.cwd()
REPO_ROOT = HERE.parents[1]                     # .../obi-one
J_DROSOPHILA = REPO_ROOT / "examples" / "J_drosophila"
CIRCUIT_DIR = HERE / "sonata_circuit"
DROSOPHILA_REPO = HERE / "Drosophila_brain_model"

if not (CIRCUIT_DIR / "circuit_config.json").exists():
    if not DROSOPHILA_REPO.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/philshiu/Drosophila_brain_model.git",
            str(DROSOPHILA_REPO),
        ])
    for p in (J_DROSOPHILA, DROSOPHILA_REPO):
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
    import drosophila_to_brian2_sonata as dros_sonata
    from model import default_params

    SUGAR_NODES = [
        720575940611875570, 720575940612670570, 720575940616885538,
        720575940617000768, 720575940617937543, 720575940620900446,
        720575940621502051, 720575940621754367, 720575940624963786,
        720575940628853239, 720575940629176663, 720575940630233916,
        720575940630797113, 720575940632425919, 720575940632889389,
        720575940633143833, 720575940637568838, 720575940638202345,
        720575940639198653, 720575940639332736, 720575940640649691,
    ]
    # FlyWire annotations (xyz positions + cell-class columns) for the 630
    # materialization (flywire_annotations v1.0.0). Merged into the SONATA nodes
    # by drosophila_to_brian2_sonata. Downloaded once into the cloned repo.
    annotations_path = DROSOPHILA_REPO / "Supplemental_file1_neuron_annotations_630.tsv"
    if not annotations_path.exists():
        annotations_url = (
            "https://github.com/flyconnectome/flywire_annotations/raw/"
            "847a711ce3b6e3cc675cf9ef9c843ba564bba1b5/"
            "supplemental_files/Supplemental_file1_annotations.tsv"
        )
        print(f"Downloading FlyWire annotations -> {annotations_path}")
        urllib.request.urlretrieve(annotations_url, annotations_path)

    dros_sonata.convert(
        CIRCUIT_DIR,
        path_comp=DROSOPHILA_REPO / "2023_03_23_completeness_630_final.csv",
        path_connnections=DROSOPHILA_REPO / "2023_03_23_connectivity_630_final.parquet",
        sugar_nodes=SUGAR_NODES,
        default_params=default_params,
        annotations_path=annotations_path,
    )

# Mandatory: SONATA circuit folder containing circuit_config.json
circuit_path = str(CIRCUIT_DIR)

# Optional: Pre-existing image files (skip generation if provided)
overview_image_path = None       # Path to overview image (.png or .webp)
sim_designer_image_path = None   # Path to simulation designer image (.png)

print(f"circuit_path: {circuit_path}")

circuit_path: /Users/james/Documents/obi/code/obi-main/obi-one/examples/J_drosophila/sonata_circuit


## Get DB client

Connects to **staging**. `obi_auth` caches the last sign-in on disk and
`get_token()` silently reuses it, so the cached staging token is cleared first
to pick up a freshly signed-in account.

In [6]:
import obi_one as obi
from entitysdk import Client, ProjectContext, models
from obi_auth import get_token
from obi_auth.config import settings as obi_auth_settings
from obi_auth.storage import Storage as ObiAuthStorage
from obi_auth.typedef import DeploymentEnvironment

ObiAuthStorage(
    config_dir=obi_auth_settings.config_dir,
    environment=DeploymentEnvironment.staging,
).clear()

# Staging "Shared results + tests" project
token = get_token(environment="staging")
project_context = ProjectContext(
    virtual_lab_id=obi.LAB_ID_STAGING_TEST,
    project_id=obi.PROJECT_ID_STAGING_TEST,
)
client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("connected to staging entitycore")

connected to staging entitycore


## Run circuit registration

The high-level `register_circuit_from_metadata` helper resolves all linked
entities from the metadata, validates the SONATA circuit, computes counts /
scale / properties, registers the circuit entity together with its derivation,
contribution and publication links, and generates + uploads all assets.

First do a **dry run** (`dry_run = True`) to check the circuit files, metadata,
and dependencies without registering anything:
- A valid SONATA circuit with `circuit_config.json` must exist.
- The circuit name must not yet exist in entitycore (to avoid duplicates).
- All linked entities must already exist in entitycore.

If the dry run succeeds, re-run with `dry_run = False` to perform the actual
registration. Set `authorized_public = True` to register a public entity
(requires a license); otherwise it is private within the chosen project.

In [7]:
from obi_one.utils.circuit_registration import register_circuit_from_metadata

In [8]:
dry_run = True             # If True, runs all checks but no actual registration
authorized_public = False  # If True, registered as public entity; otherwise private

In [9]:
# Create & register circuit entity (+ links + assets)
registered_circuit = register_circuit_from_metadata(
    client=client,
    circuit_metadata=circuit_metadata,
    circuit_path=circuit_path,
    contributions=circuit_contributions,
    publications=circuit_publications,
    authorized_public=authorized_public,
    overview_image_path=overview_image_path,
    sim_designer_image_path=sim_designer_image_path,
    dry_run=dry_run,
)

No license specified!
Connectivity matrix asset generation/registration failed: Unknown properties: [np.str_('etype'), np.str_('layer'), np.str_('mtype')]


After registration, the registered Circuit entity can be checked.

In [10]:
if registered_circuit:
    # Fetch entity incl. assets
    res = client.get_entity(entity_id=registered_circuit.id, entity_type=models.Circuit)

    # Print circuit info
    print(f"Circuit '{res.name}' registered (ID {res.id})")
    print(f"  scale={res.scale}, neurons={res.number_neurons}, synapses={res.number_synapses}, connections={res.number_connections}")
    print(f"  has_morphologies={res.has_morphologies}, has_point_neurons={res.has_point_neurons}, has_electrical_cell_models={res.has_electrical_cell_models}, has_spines={res.has_spines}")

    # Print asset info
    print(f"Assets registered ({len(res.assets)}):")
    for a in res.assets:
        print(f"  Asset '{a.label}' ({'dir' if a.is_directory else 'file'}) registered (ID {a.id})")
else:
    print("Circuit not registered!")

Circuit not registered!
